# 01 分箱模块 (core.binning)

覆盖所有分箱算法、自定义切分点、单调约束、批量分箱、转换模式。

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import numpy as np, pandas as pd
import matplotlib; matplotlib.use('Agg')
import hscredit as hsc
from hscredit import germancredit, init_setting, OptimalBinning
from hscredit import (
    UniformBinning, QuantileBinning, TreeBinning, ChiMergeBinning,
    BestKSBinning, BestIVBinning, MDLPBinning, CartBinning,
    KMeansBinning, MonotonicBinning, SmoothBinning,
    BestLiftBinning, TargetBadRateBinning,
)
init_setting()
df = germancredit()
target = 'creditability'
y = df[target]
print(df.shape)
df.head(2)

## 1. 快速开始 — OptimalBinning 统一入口

In [ ]:
feats = ['duration.in.month', 'credit.amount', 'age.in.years']
binner = OptimalBinning(method='mdlp', max_n_bins=5)
binner.fit(df[feats], y)
for f, sp in binner.splits_.items():
    print(f'{f}: {sp}')
binner.bin_tables_['duration.in.month'][['分箱标签','样本总数','坏样本率','分档IV值']]

## 2. 所有分箱方法对比

In [ ]:
feat = 'duration.in.month'
X1 = df[[feat]]
methods = {
    'uniform': UniformBinning(n_bins=5),
    'quantile': QuantileBinning(n_bins=5),
    'tree': TreeBinning(max_n_bins=5),
    'chi': ChiMergeBinning(max_n_bins=5),
    'best_ks': BestKSBinning(max_n_bins=5),
    'best_iv': BestIVBinning(max_n_bins=5),
    'mdlp': MDLPBinning(max_n_bins=5),
    'cart': CartBinning(max_n_bins=5),
    'kmeans': KMeansBinning(n_bins=5),
    'smooth': SmoothBinning(max_n_bins=5),
    'best_lift': BestLiftBinning(max_n_bins=5),
    'target_bad_rate': TargetBadRateBinning(max_n_bins=5),
}
rows = []
for name, b in methods.items():
    try:
        b.fit(X1, y)
        bt = b.bin_tables_.get(feat, pd.DataFrame())
        iv_val = bt['分档IV值'].sum() if '分档IV值' in bt.columns else 0
        rows.append({'方法': name, '分箱数': len(bt), 'IV': round(iv_val,4)})
    except Exception as e:
        rows.append({'方法': name, '分箱数': 0, 'IV': str(e)[:40]})
pd.DataFrame(rows)

## 3. 自定义分割点

In [ ]:
b_custom = OptimalBinning(user_splits={'duration.in.month': [12, 24, 36, 48]})
b_custom.fit(df[['duration.in.month']], y)
b_custom.bin_tables_['duration.in.month'][['分箱标签','样本总数','坏样本率','分档IV值']]

## 4. 单调性约束

In [ ]:
for mono in ['ascending','descending','peak','valley','auto']:
    try:
        b = OptimalBinning(method='monotonic', monotonic=mono, max_n_bins=6)
        b.fit(df[['credit.amount']], y)
        rates = b.bin_tables_['credit.amount']['坏样本率'].round(3).tolist()
        print(f'{mono:12s}: {rates}')
    except Exception as e:
        print(f'{mono:12s}: {e}')

## 5. 类别型特征分箱

In [ ]:
b_cat = OptimalBinning(method='tree', max_n_bins=4)
b_cat.fit(df[['purpose']], y)
b_cat.bin_tables_['purpose'][['分箱标签','样本总数','坏样本率','分档IV值']]

## 6. 批量分箱

In [ ]:
num_feats = df.select_dtypes('number').columns.drop(target).tolist()
b_batch = OptimalBinning(method='mdlp', max_n_bins=5)
b_batch.fit(df[num_feats], y)
pd.DataFrame([{'特征':f,'IV':round(bt['分档IV值'].sum(),4),'箱数':len(bt)}
    for f,bt in b_batch.bin_tables_.items() if '分档IV值' in bt.columns
]).sort_values('IV',ascending=False)

## 7. 转换模式（WOE / indices / labels）

In [ ]:
b3 = OptimalBinning(method='mdlp', max_n_bins=5)
b3.fit(df[num_feats[:3]], y)
print('WOE:'); print(b3.transform(df[num_feats[:3]], metric='woe').head(3))
print('idx:'); print(b3.transform(df[num_feats[:3]], metric='indices').head(3))
print('lbl:'); print(b3.transform(df[num_feats[:3]], metric='labels').head(3))

## 8. auto_select_method

In [ ]:
feat = 'credit.amount'
best = OptimalBinning.auto_select_method(df[[feat]], y, feat)
print(f'最优方法: {best}')
b_auto = OptimalBinning(method=best, max_n_bins=5)
b_auto.fit(df[[feat]], y)
b_auto.bin_tables_[feat][['分箱标签','样本总数','坏样本率','分档IV值']]